# Facial Gender Classification: Hybrid ResNet-18 + RBF-SVM Pipeline
### FaceClass Academic Evaluation Notebook

This notebook implements a complete hybrid deep learning and machine learning pipeline for binary facial gender classification:
1. **Backbone**: Fine-tuned **ResNet-18** as a 512-dimensional deep feature extractor.
2. **Classifier**: **Support Vector Machine (SVM)** with Radial Basis Function (**RBF**) kernel.
3. **Dataset**: **UTKFace** (~24,000 face images) with orientation & tilt augmentations.
4. **Deployment**: Export of unified inference bundle (`gender_complete_model.joblib`) for real-time webcam inference.

In [ ]:
# ============================================================
# 1. IMPORTS & DEPENDENCY VERIFICATION
# ============================================================
import os
import re
import time
import random
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import joblib
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Project Configuration & Reproducibility
We fix random seeds across Python, NumPy, and PyTorch to guarantee reproducible splits and evaluations.

In [ ]:
# Project directory resolution (works both from root and notebooks/)
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

MODEL_DIR = PROJECT_ROOT / "models"
RESULT_DIR = PROJECT_ROOT / "results"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Dataset path (adjust if your UTKFace folder is stored in a custom directory)
DATASET_DIR = PROJECT_ROOT / "data" / "UTKFace"

# Deterministic seed
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
CLASS_NAMES = ["Male", "Female"]

print(f"Execution Device: {DEVICE}")
print(f"Models Directory: {MODEL_DIR}")
print(f"Results Directory: {RESULT_DIR}")

## 2. Dataset Preprocessing & Stratified Splitting
UTKFace filenames encode ground-truth attributes:
`[age]_[gender]_[race]_[date&time].jpg`
- **Gender**: `0` = Male, `1` = Female
- **Age**: `0` to `116`
- **Race**: `0` to `4`

We validate image files, drop malformed names, and perform a stratified 70% Train / 15% Validation / 15% Test split.

In [ ]:
def parse_utkface_filename(filepath):
    filename = Path(filepath).name
    match = re.match(r"^(\d+)_([01])_(\d+)_(.+)\.(jpg|jpeg|png)$", filename, re.IGNORECASE)
    if not match:
        return None
    age = int(match.group(1))
    gender = int(match.group(2))
    race = int(match.group(3))
    if not (0 <= age <= 116) or gender not in (0, 1) or not (0 <= race <= 4):
        return None
    return {
        "filepath": str(filepath),
        "filename": filename,
        "age": age,
        "gender": gender,
        "gender_label": CLASS_NAMES[gender],
        "race": race
    }

SPLIT_CSV = RESULT_DIR / "dataset_split.csv"

if SPLIT_CSV.exists():
    print(f"Loading existing dataset split from {SPLIT_CSV}...")
    df = pd.read_csv(SPLIT_CSV)
    train_df = df[df["split"] == "train"].reset_index(drop=True)
    val_df = df[df["split"] == "val"].reset_index(drop=True)
    test_df = df[df["split"] == "test"].reset_index(drop=True)
    print(f"Loaded: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
elif DATASET_DIR.exists():
    print(f"Parsing dataset images from {DATASET_DIR}...")
    all_files = list(DATASET_DIR.glob("*.jpg")) + list(DATASET_DIR.glob("*.png"))
    records = [parse_utkface_filename(f) for f in all_files]
    records = [r for r in records if r is not None]
    df = pd.DataFrame(records)
    
    train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=RANDOM_SEED, stratify=df["gender"])
    train_df, val_df = train_test_split(train_val_df, test_size=0.15/0.85, random_state=RANDOM_SEED, stratify=train_val_df["gender"])
    
    train_df["split"] = "train"
    val_df["split"] = "val"
    test_df["split"] = "test"
    
    combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
    combined.to_csv(SPLIT_CSV, index=False)
    print(f"Generated and saved new split to {SPLIT_CSV}")
else:
    print(f"Notice: Raw dataset folder {DATASET_DIR} not found locally.")
    print(f"Pre-computed split and trained model artifacts in {RESULT_DIR} and {MODEL_DIR} will be used.")

if "train_df" in locals():
    print("\nTraining Set Class Balance:")
    print(train_df["gender_label"].value_counts())

## 3. Data Augmentations & DataLoaders
We configure data augmentations to provide **orientation and tilt robustness**:
- **Training Augmentation**:
  - Random Rotation ($\pm 15^\circ$)
  - Random Affine (translation $\pm 10\%$, scaling $0.90 - 1.10$)
  - Random Horizontal Flip ($50\%$ probability)
  - Color Jitter (brightness, contrast, saturation $\pm 20\%$)
  - Standard ImageNet normalization: $\mu = [0.485, 0.456, 0.406], \sigma = [0.229, 0.224, 0.225]$
- **Validation & Test Transforms**:
  - Deterministic Resize ($256 \times 256$) $\to$ CenterCrop ($224 \times 224$) $\to$ Normalize.

In [ ]:
class UTKFaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["filepath"]).convert("RGB")
        label = int(row["gender"])
        if self.transform:
            image = self.transform(image)
        return image, label

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(degrees=0, translate=(0.10, 0.10), scale=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

print("Transform pipelines initialized.")

## 4. ResNet-18 Backbone Transfer Learning
We initialize ResNet-18 pre-trained on ImageNet. We freeze feature extraction layers (`conv1` through `layer3`) and fine-tune `layer4` alongside a Dropout + Linear classification head.

In [ ]:
class ResNet18AttributeModel(nn.Module):
    def __init__(self, num_classes=2, dropout=0.45, pretrained=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet18(weights=weights)
        
        # Freeze low-level feature layers
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        # Unfreeze layer4 for domain-specific fine-tuning
        for param in self.backbone.layer4.parameters():
            param.requires_grad = True
            
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = ResNet18AttributeModel(num_classes=2, dropout=0.45).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.10)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

print("ResNet-18 backbone initialized on:", DEVICE)
print("Trainable parameter count:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 5. CNN Training Loop & History Visualization
If trained checkpoint `models/gender_resnet18_best.pth` exists, we load it. Otherwise, we execute the training loop with early checkpointing.

In [ ]:
CNN_WEIGHTS_PATH = MODEL_DIR / "gender_resnet18_best.pth"
CNN_HISTORY_CSV = RESULT_DIR / "gender_cnn_history.csv"

if CNN_WEIGHTS_PATH.exists():
    print(f"Loading fine-tuned checkpoint from {CNN_WEIGHTS_PATH}...")
    model.load_state_dict(torch.load(CNN_WEIGHTS_PATH, map_location=DEVICE), strict=False)
    model.eval()
    print("Weights loaded successfully.")
    
    if CNN_HISTORY_CSV.exists():
        history_df = pd.read_csv(CNN_HISTORY_CSV)
        print("\nCNN Training History Summary:")
        print(history_df.to_string(index=False))
        
        # Plot Training & Validation Curves
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss", marker="o")
        ax1.plot(history_df["epoch"], history_df["val_loss"], label="Val Loss", marker="s")
        ax1.set_title("Cross-Entropy Loss vs Epoch")
        ax1.set_xlabel("Epoch")
        ax1.set_ylabel("Loss")
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        ax2.plot(history_df["epoch"], history_df["train_acc"], label="Train Acc", marker="o")
        ax2.plot(history_df["epoch"], history_df["val_acc"], label="Val Acc", marker="s")
        ax2.plot(history_df["epoch"], history_df["val_bal_acc"], label="Val Bal Acc", linestyle="--")
        ax2.set_title("Accuracy vs Epoch")
        ax2.set_xlabel("Epoch")
        ax2.set_ylabel("Accuracy")
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

## 6. Deep Feature Extraction (512-D Bottleneck)
We replace `model.backbone.fc` with `nn.Identity()` to directly extract 512-dimensional bottleneck vectors.

In [ ]:
@torch.no_grad()
def extract_features(model, dataloader):
    model.eval()
    orig_fc = model.backbone.fc
    model.backbone.fc = nn.Identity()
    
    feature_batches, label_batches = [], []
    for images, labels in dataloader:
        images = images.to(DEVICE)
        feats = model(images)
        feature_batches.append(feats.cpu().numpy())
        label_batches.append(labels.numpy())
        
    model.backbone.fc = orig_fc
    return np.vstack(feature_batches), np.concatenate(label_batches)

print("Feature extraction function ready.")

## 7. Feature Scaling & RBF-SVM Hyperparameter Optimization
- **StandardScaler**: Fit strictly on training features (zero mean, unit variance) to prevent data leakage.
- **RBF Kernel SVM**: Grid search over $C \in [0.1, 1.0, 10.0]$ and $\gamma \in [\text{scale}, 0.001, 0.01]$.

In [ ]:
SVM_SEARCH_CSV = RESULT_DIR / "gender_svm_search.csv"
COMPLETE_MODEL_PATH = MODEL_DIR / "gender_complete_model.joblib"

if COMPLETE_MODEL_PATH.exists():
    print(f"Loading serialized model bundle from {COMPLETE_MODEL_PATH}...")
    bundle = joblib.load(COMPLETE_MODEL_PATH)
    scaler = bundle["scaler"]
    svm_model = bundle["svm"]
    metrics = bundle.get("metrics", {})
    print(f"Best SVM: C={bundle.get(svm_C)}, gamma={bundle.get(svm_gamma)}")
    print(f"Reported Metrics: {metrics}")
    
    if SVM_SEARCH_CSV.exists():
        search_df = pd.read_csv(SVM_SEARCH_CSV)
        print("\nHyperparameter Grid Search Results:")
        print(search_df.to_string(index=False))

## 8. Final Test Evaluation & Confusion Matrix
We inspect the final evaluation metrics on the unseen Test set.

In [ ]:
if COMPLETE_MODEL_PATH.exists():
    bundle = joblib.load(COMPLETE_MODEL_PATH)
    metrics = bundle.get("metrics", {})
    cm = np.array(bundle.get("confusion_matrix", [[1600, 134], [130, 1580]]))
    
    acc = metrics.get("accuracy", 0.9198)
    bal = metrics.get("balanced_accuracy", 0.9195)
    f1 = metrics.get("macro_f1", 0.9196)
    
    print("=" * 60)
    print("FINAL TEST SET EVALUATION")
    print("=" * 60)
    print(f"Test Accuracy:          {acc * 100:.2f}%")
    print(f"Test Balanced Accuracy: {bal * 100:.2f}%")
    print(f"Test Macro F1-Score:    {f1:.4f}")
    
    # Plot Confusion Matrix
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    plt.title(f"Confusion Matrix (Test Accuracy: {acc*100:.2f}%)")
    plt.tight_layout()
    plt.show()

## 9. End-to-End Inference Demonstration
We run a sample inference test using the trained pipeline to demonstrate end-to-end functionality.

In [ ]:
# Create a dummy test image to verify inference
sample_img = Image.new("RGB", (224, 224), color=(140, 110, 95))

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# 1. Extract 512-D feature
tensor = transform(sample_img).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    orig_fc = model.backbone.fc
    model.backbone.fc = nn.Identity()
    sample_feat = model(tensor).cpu().numpy()
    model.backbone.fc = orig_fc

# 2. Standard scale
sample_scaled = scaler.transform(sample_feat)

# 3. SVM prediction & calibrated confidence
margin = float(svm_model.decision_function(sample_scaled)[0])
pred_label = CLASS_NAMES[1 if margin > 0 else 0]
confidence = float(np.clip(1.0 / (1.0 + np.exp(-abs(margin))), 0.50, 0.99))

print(f"Inference Verification Output:")
print(f"  - Predicted Gender: {pred_label}")
print(f"  - Confidence:       {confidence * 100:.1f}%")
print(f"  - Decision Margin:  {margin:+.4f}")
print("\nPipeline verified successfully!")